In [0]:
%sql

DESCRIBE bakery_pos_develop.bronze.res_users

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_res_users AS
SELECT 
    id AS user_id,
    name AS user_name,
    CAST(create_date AS TIMESTAMP) AS created_at,
    CAST(write_date AS TIMESTAMP) AS updated_at,
    processing_timestamp AS bronze_ingested_at
FROM bakery_pos_develop.bronze.res_users
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY id
    ORDER BY write_date DESC,
    processing_timestamp DESC        
) = 1


In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.silver.res_users (
    user_id INT,
    user_name STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    bronze_ingested_at TIMESTAMP,
    silver_processed_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN
) USING DELTA;



In [0]:
%sql

MERGE INTO bakery_pos_develop.silver.res_users AS target
USING (
  SELECT *, current_timestamp() AS silver_processed_at
  FROM stg_res_users
) AS source
ON target.user_id = source.user_id
WHEN MATCHED AND source.updated_at > target.updated_at THEN
  UPDATE SET
    user_name = source.user_name,
    created_at = source.created_at,
    updated_at = source.updated_at,
    is_updated = True,
    is_deleted = False,
    bronze_ingested_at = source.bronze_ingested_at,
    silver_processed_at = source.silver_processed_at
WHEN NOT MATCHED THEN
  INSERT (
    user_id,
    user_name,
    created_at,
    updated_at,
    bronze_ingested_at,
    silver_processed_at,
    is_updated,
    is_deleted
  )
  VALUES (
    source.user_id,
    source.user_name,
    source.created_at,
    source.updated_at,
    source.bronze_ingested_at,
    source.silver_processed_at,
    False,
    False
  );